____
### 0. Preamble
This notebook serves to create a Recurrent Neural Network (RNN) autoregression model to solve the SQL autocomplete problem

____
### 1. Imports

In [5]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

In [6]:
from autoregression.data.dataprep import N_TOKENS, PAD_ID, EOS_ID, SOS_ID, TOKEN_TO_ID, ID_TO_TOKEN, collate
from autoregression.data.dataprep import SQLSeqDataset

____
### 2. The Recurrent Neural Network (RNN) approach
- RNN adds a hidden state that gets updated at each time step and carries information forward.
- At each step `t`, the RNN does:

${h_t} = tanh(W_h · h_{(t-1)} + W_x · x_t + b)$

${y_t} = {W_y} · {h_t}$

- where: 
    1. ${x_t}$: input at time t
    2. ${h_{(t-1)}}$ : hidden state from the previous step (the "memory" accumulated)
    3. ${h_t}$ : new hidden state
    4. ${y_t}$ : prediction, the value at t+1

- in this autoregression problem, the sequence is fed in one step at a time 
- hidden state at time t, ${h_t}$, is a compressed summary of everything seen so far

#### 2.1 Weakness of default RNNs
- Over long sequences, the hidden state trivialises the data from the earlier timesteps due to gradients shrinking / exploding
- Therefore, plain RNNs struggle to remember anything more than 10-20 steps, counting from where it currently is.

#### 2.2 Addressing the weakness (LSTM)
- A Long Short-Term Memory(LSTM) RNN solves the problem of 'forgetting' by using a more elaborate mechanism involving cell states (long-term memory) and 3 gates that control information flow
- The 3 gates are: 
    1. Forget Gate : decides what to throw away from the cell state

        $f_t = σ(W_f · [h_{(t-1)}, x_t] + b_f)$
    2. Input Gate : decides what new information to store
        
        $i_t = σ(W_i · [h_{(t-1)}, x_t] + b_i)$
    3. Output Gate: decides what to output based on the cell state

        $o_t = σ(W_o · [h_{(t-1)}, x_t] + b_o)$

- The cell state ${C_t}$ acts like a conveyor belt, information can flow along it largely unchanged unless a gate decides to modify it. 
- This lets gradients flow back much further without vanishing, so LSTMs can learn longer-range dependencies than plain RNNs.
- this process is given by:
    1. Forget gate applies the changes into cell state to decide if anything should be forgotten
        - looks at $h_{t-1}$ and $x_t$ and decides what to throw away from the old cell state 
        - outputs values near 0 are "forgotten" and those near 1 are kept.
    2. Input gate receives new info and filters what to let in 
        - processes the $x_t$ before passing it to the Candidate layer
    3. Candidate layer creates a vector of new candidate values $C̃_t$ that could be added to the cell state
        - candidate values calculated using : $C̃_t = tanh(W_C · [h_{(t-1)}, x_t] + b_C)$ 
        - when updating the cell state, the input gate's output, $i_t$ , scales down the candidate values : $i_t * C̃_t$
        - at the same time, the forget gate's output scales down the previous cell state: $f_t * C_{(t-1)}$ , which essentially helps to drop what is deemed irrelevant 
        - the old cell state is then combined with the new info: $C_t = f_t * C_{(t-1)} + i_t * C̃_t$
    4. Output gate decides what part of the updated cell state $C_t$ to be exposed as output
        - filters $C_t$ through a sigmoid to produce $o_t$ : $o_t = σ(W_o · [h_{(t-1)}, x_t] + b_o)$ 
    5. Hidden state computed
        - the cell state is squashed through a `tanh` function and multiplied by the output gate's results $o_t$ to give the new hidden state : $h_t = o_t * tanh(C_t)$ 

____
### 3. Implementation

#### 3.1 LSTMCore
- the engine that powers the model

In [7]:
class LSTMCore(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers, dropout, pad_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, vocab_size)
 
    def forward(self, input_ids, hidden=None):
        x = self.embedding(input_ids) # (B, T, emb_dim)
        out, hidden = self.lstm(x, hidden) # (B, T, hidden_dim)
        out = self.dropout(out)
        logits = self.head(out) # (B, T, vocab)
        return logits, hidden


##### 3.11 Initialisation
```python
self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
self.lstm = nn.LSTM(
    input_size=emb_dim,
    hidden_size=hidden_dim,
    num_layers=num_layers,
    batch_first=True,
    dropout=dropout if num_layers > 1 else 0.0,
    )
self.dropout = nn.Dropout(dropout)
self.head = nn.Linear(hidden_dim, vocab_size)
```
- `self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)` : `nn.Embedding` is a lookup table that maps integer token IDs to dense vectors
    - this removes the need to feed in raw int IDs for each SQL block, removing the implied relationship between SQL blocks due to the numbers being close / far from each other
- `nn.LSTM(...)` : creating the lstm neural network
    - `input_size=emb_dim` : size of each timestep's input vector, i.e. the embedding dimension.
    - `hidden_size=hidden_dim` : size of the hidden state used, must be sufficiently big to capture the full complexity of the hidden state (ie, large enough to store "memory")
    - `num_layers=num_layers` : stack multiple LSTM layers on top of each other, increasing layers = increasing representation depth 
    - `batch_first=True` : controls tensor shape convention, `True` makes input/output shape be `(batch, seq_len, features)`
    - `dropout=dropout if num_layers > 1 else 0.0` : turns dropout on, which randomly sets a fraction of a layer's output values to zero on each forward pass
        - a network can end up leaning heavily on a small number of units to do the work, causing the NN to memorise patterns and essentially "overfit" 
        - turning dropout on forces the neural network to avoid relying on any single neuron from doing the heavy lifting
- `self.dropout = nn.Dropout(dropout)` : a separate dropout applied after the LSTM's final output, before the linear head.
    - this is distinct from the LSTM's internal inter-layer dropout, this one hits the final (B, T, hidden_dim) output.
    - this is done to make sure the final layer receives the zeroing treatment as well before being fed into `self.head`
- `self.head = nn.Linear(hidden_dim, vocab_size)` : projection from hidden-state space back to vocabulary space
    - takes each timestep's `hidden_dim`-sized vector and maps it to `vocab_size` raw scores (in the form of logits)
    - one score per possible next token is generated 
    - what CrossEntropyLoss (or a softmax, at generation time) operates on to decide what token comes next
    - turns "internal representation" into "prediction over the vocabulary"

##### 3.12 `forward()`
```python
def forward(self, input_ids, hidden=None):
    x = self.embedding(input_ids)
    out, hidden = self.lstm(x, hidden) # (B, T, hidden_dim)
    out = self.dropout(out)
    logits = self.head(out) # (B, T, vocab)
    return logits, hidden
```
- `forward(self, input_ids, hidden=None)` : method to produce the next output after feeding into the LSTM
    - `input_ids`: a batch of token-id sequences, shaped `(batch, seq_len)`, ints straight from `collate_fn`
    - `hidden=None` : optional hidden state, `(h_0, c_0)` tuple used to seed the LSTM's starting hidden/cell state. Defaults to None since the NN starts with no hidden state  This parameter is the whole reason sample() can generate token-by-token 
- `x = self.embedding(input_ids)` : process the input using `self.embeddings`
- `out, hidden = self.lstm(x, hidden)` : feeds the input `x` and hidden vector `hidden` to obtain the output `out` and the new hidden state 
- `out = self.dropout(out)` : applies the external dropout function to the last output
- `logits = self.head(out)` : maps the output to a tensor storing the logit value of each block 


#### 3.2 RNNAgent
- class that owns the LSTMCore

In [ ]:
class RNNAgent:
    def __init__(self, vocab_size=N_TOKENS, pad_id=PAD_ID, sos_id=SOS_ID,eos_id=EOS_ID,
        emb_dim=128, hidden_dim=256, num_layers=2, dropout=0.2, lr=1e-3, device=None):
        self.pad_id = pad_id
        self.sos_id = sos_id
        self.eos_id = eos_id
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = LSTMCore(
            vocab_size=vocab_size,
            emb_dim=emb_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            pad_id=pad_id,
        ).to(self.device)
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)
        self.criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
 
    def fit(self, training_tensors, validation_tensors, collate_fn,
            epochs=20, batch_size=32, verbose=True):
        train_loader = DataLoader(
            SQLSeqDataset(training_tensors), batch_size=batch_size, shuffle=True,
            collate_fn=collate_fn,
        )
        history = {"training pplx": [], "validation pplx": []}
        for epoch in range(1, epochs + 1):
            train_ppl = self.run_epoch(train_loader, train=True)
            val_ppl = self.evaluate(validation_tensors, collate_fn, batch_size=batch_size)
 
            history["training pplx"].append(train_ppl)
            history["validation pplx"].append(val_ppl)
            if verbose:
                print(f"epoch {epoch:2d} | training pplx {train_ppl:7.3f} | validation pplx {val_ppl:7.3f}")
        return history
 
    def run_epoch(self, loader, train):
        self.model.train(mode=train)
        total_loss, total_tokens = 0.0, 0
 
        context = torch.enable_grad() if train else torch.no_grad()
        with context:
            for input_ids, target_ids in loader:
                input_ids = input_ids.to(self.device)
                target_ids = target_ids.to(self.device)
                if train:
                    self.optimizer.zero_grad()
                logits, _ = self.model(input_ids)
                loss = self.criterion(
                    logits.reshape(-1, logits.size(-1)),
                    target_ids.reshape(-1),
                )
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                    self.optimizer.step()
                n_real_tokens = (target_ids != self.pad_id).sum().item()
                total_loss += loss.item() * n_real_tokens
                total_tokens += n_real_tokens
        return math.exp(total_loss / total_tokens)
 
    def evaluate(self, sequences, collate_fn, batch_size=32, n=1, verbose=False):
        loader = DataLoader(
            SQLSeqDataset(sequences), batch_size=batch_size, shuffle=False,
            collate_fn=collate_fn,
        )
        pplx_lst = []
        for _ in range (0,n):
            pplx = self.run_epoch(loader, train=False)
            if n == 1:
                return pplx
            else:
                pplx_lst.append(pplx)
        avg_pplx = sum(pplx_lst)/len(pplx_lst)
        if verbose:
            print(f"Average perplexity over {n} runs: {round(avg_pplx,2)}")
        return avg_pplx

    @torch.no_grad()
    def sample(self, id_to_token=None, max_len=40, temperature=1.0):
        self.model.eval()
        input_id = torch.tensor([[self.sos_id]], device=self.device)
        hidden = None
        generated = [self.sos_id]
        for _ in range(max_len):
            logits, hidden = self.model(input_id, hidden)          # (1, 1, vocab)
            probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)      # (1, 1)
            token = next_id.item()
            generated.append(token)
            if token == self.eos_id:
                break
            input_id = next_id  # feed only the new token; hidden carries the past
 
        if id_to_token is not None:
            return [id_to_token[i] for i in generated]
        return generated
 
    def save(self, path):
        torch.save(self.model.state_dict(), path)
 
    def load(self, path):
        self.model.load_state_dict(torch.load(path, map_location=self.device))


##### 3.21 Initialisation
```python
self.pad_id = pad_id
self.sos_id = sos_id
self.eos_id = eos_id
self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
self.model = LSTMCore(
    vocab_size=vocab_size,
    emb_dim=emb_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    dropout=dropout,
    pad_id=pad_id,
    ).to(self.device)
self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)
self.criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
```
- `self.model = LSTMCore(...)` : initialises the LSTM core with the input parameters
- `self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)` : registers the model's parameters inside the Adam optimiser
    - this is what allows gradients to flow through all parameters during the training process
- `self.criterion = nn.CrossEntropyLoss(ignore_index=pad_id)` : the loss function used for back propagation
    - `ignore_index=pad_id` tells the loss to skip positions where `pad_id` is present
    - this is necessary as a large portion of the sequence might be padding, especially early on

##### 3.22 `run_epoch()`
```python
def run_epoch(self, loader, train):
    self.model.train(mode=train)
    total_loss, total_tokens = 0.0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for input_ids, target_ids in loader:
            input_ids = input_ids.to(self.device)
            target_ids = target_ids.to(self.device)
            if train:
                self.optimizer.zero_grad()
            logits, _ = self.model(input_ids)
            loss = self.criterion(
                logits.reshape(-1, logits.size(-1)),
                target_ids.reshape(-1)
            )
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
            n_real_tokens = (target_ids != self.pad_id).sum().item()
            total_loss += loss.item() * n_real_tokens
            total_tokens += n_real_tokens
    return math.exp(total_loss / total_tokens)
```

- `self.model.train(mode=train)` : sets the model's mode accordingly
    - `train=True` turns the model to training mode `model.train()`
    - `train=False` turns the model to evaluation mode `model.eval()`
- `total_loss, total_tokens = 0.0, 0` : initialising empty values first
- `context = torch.enable_grad() if train else torch.no_grad()` : enables or disables gradients
    - needed to for reusability for training or evaluation
- `with context` : becomes `with torch.enable_grad()` for training and `with torch.no_grad()` for evaluation
- `for input_ids, target_ids in loader`, `input_ids = input_ids.to(self.device)` and `target_ids = target_ids.to(self.device)` : iterates through the `DataLoader` object passed in and moving the tensors to the device for training
- `if train: self.optimizer.zero_grad()` : clears gradients from the previous batch 
- `logits, _ = self.model(input_ids)` : runs the forward pass
    - feeds the input `input_id` from the batch into the LSTM model `self.model`
    - stores the output as `logits`
- `loss = self.criterion(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1))` : calculates the loss value using the logits and the target
    - `.reshape` is called to flatten the 2 inputs as they `CrossEntropyLoss` expects 2D input `(N, vocab)` and 1D target `(N, )`
- `if train, loss.backward()` : backpropagation, computes gradients for every parameter
- `torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)` : rescales gradients if their combined norm exceeds 1.0
    - this is to prevent gradient changes from being to large and destabilise training
    - this is important in RNNs such as LSTM which are prone to exploding gradients
- `self.optimizer.step()` : applies the update to the weights using the gradients computed

- `n_real_tokens = (target_ids != self.pad_id).sum().item()` : filters out the tokens that are padding
- `total_loss += loss.item() * n_real_tokens` : adds the batch's total loss 
- `total_tokens += n_real_tokens` : adds the number of real tokens for tracking
- `math.exp(total_loss / total_tokens)` 
    - `total_loss/total_tokens` is the epochs avg cross-entropy loss per real token
    - applying `math.exp()` converts it to perplexity, roughly interpretable as "how many tokens the mode was on avg uncertain between", the lower the better 
    - perplexity = 1 means the model was perfectly confident

##### 3.23 `evaluate()`
```python
def evaluate(self, sequences, collate_fn, batch_size=32, n=1, verbose=False):
    loader = DataLoader(
        SQLSeqDataset(sequences), batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn,
    )
    pplx_lst = []
    for _ in range (0,n):
        pplx = self.run_epoch(loader, train=False)
        if n == 1:
            return pplx
        else:
            pplx_lst.append(pplx)
    avg_pplx = sum(pplx_lst)/len(pplx_lst)
    if verbose:
        print(f"Average perplexity over {n} runs: {avg_pplx}")
    return avg_pplx
```
- runs 1 epoch under evaluation mode by default, returning the perplexity
- else, run `n` and return the perplexity over `n` runs

##### 3.24 `fit()`
```python
def fit(self, training_tensors, validation_tensors, collate_fn,
    epochs=20, batch_size=32, verbose=True):
    train_loader = DataLoader(SQLSeqDataset(training_tensors), 
        batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    history = {"train_ppl": [], "val_ppl": []}
    for epoch in range(1, epochs + 1):
        train_ppl = self.run_epoch(train_loader, train=True)
        val_ppl = self.evaluate(validation_tensors, collate_fn, batch_size=batch_size)
        history["train_ppl"].append(train_ppl)
        history["val_ppl"].append(val_ppl)
        if verbose:
            print(f"epoch {epoch:2d} | train ppl {train_ppl:7.3f} | val ppl {val_ppl:7.3f}")
    return history
```

- `train_loader = DataLoader(SQLSeqDataset(training_tensors), batch_size=batch_size, shuffle=True, collate_fn=collate_fn)` : creating the `DataLoader` object
    - `SQLSeqDataset(training_tensors)` creates the dataset object and sends it into the `Dataloader` object directly 
- `history = {"train_ppl": [], "val_ppl": []}` : initialise a dictionary to store values
    - `train_ppl` stores the perplexity value for the training set and `val_ppl` stores the perplexity value for the validation set, throughout the epochs
- `for epoch in range(1, epochs + 1):` iterate for the number of epochs indicated
- `train_ppl = self.run_epoch(train_loader, train=True)` : runs 1 epoch under training mode, collecting the peplexity value
- `val_ppl = self.evaluate(validation_tensors, collate_fn, batch_size=batch_size)` : runs 1 epoch under evaluations mode, measuring how well how the model generalizes right now
- `history["train_ppl"].append(train_ppl)` and `history["val_ppl"].append(val_ppl)` : store perplexity values for the epoch
- `if verbose: ...` : printing for logging

##### 3.25 `sample()`
```python
@torch.no_grad()
def sample(self, id_to_token=None, max_len=40, temperature=1.0):
    self.model.eval()
    input_id = torch.tensor([[self.sos_id]], device=self.device)
    hidden = None
    generated = [self.sos_id]
    for _ in range(max_len):
        logits, hidden = self.model(input_id, hidden)          # (1, 1, vocab)
        probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)      # (1, 1)
        token = next_id.item()
        generated.append(token)
        if token == self.eos_id:
            break
        input_id = next_id  # feed only the new token; hidden carries the past
    if id_to_token is not None:
        return [id_to_token[i] for i in generated]
    return generated
```
- `@torch.no_grad()` to ensure that the function is carried out without computing gradients so that the model weights do no change
- `self.model.eval()` : switches the model to evaluation mode
- `input_id = torch.tensor([[self.sos_id]], device=self.device)` : initialises a tensor with a start of sequence token `<SOS>` in the form of the id
- `hidden = None` : starts with no hidden state
- `generated = [self.sos_id]` : initialises an array to store the generated sequence later on


____
### 4. Training and Evauating the LSTM model
1. Import training and evaluation data


In [9]:
from autoregression.data.datasplit import training_tensors, validation_tensors

Audit OK: all held-out variants are safe (no token is solely sourced by a held-out variant).
train:1277 problems
interp_val: 67 problems
ood[select]:  336 problems (only 'select' is novel)
ood[where]:  384 problems (only 'where' is novel)
ood[groupby]:  336 problems (only 'groupby' is novel)
ood[orderby]:  448 problems (only 'orderby' is novel)
ood[limit]:  672 problems (only 'limit' is novel)
ood_compound: 1880 problems (2+ knobs novel at once)

  select: counts per in-distribution value = {0: 336, 1: 336, 2: 336, 3: 336}  (balanced)
  join: counts per in-distribution value = {0: 672, 1: 672}  (balanced)
  where: counts per in-distribution value = {0: 192, 1: 192, 2: 192, 3: 192, 4: 192, 5: 192}  (balanced)
  groupby: counts per in-distribution value = {0: 336, 2: 336, 3: 336, 4: 336}  (balanced)
  orderby: counts per in-distribution value = {0: 448, 2: 448, 3: 448}  (balanced)
  limit: counts per in-distribution value = {0: 672, 2: 672}  (balanced)


2. Instantiate instance of `RNNAgent`

In [19]:
rnn = RNNAgent(emb_dim=128,
               hidden_dim=256,
               num_layers=2,
               dropout=0.2,
               lr=1e-3)

3. Train and evaluate

In [20]:
data = rnn.fit(training_tensors, validation_tensors, collate_fn=collate, epochs=500)

epoch  1 | training pplx   8.475 | validation pplx   2.207
epoch  2 | training pplx   1.687 | validation pplx   1.455
epoch  3 | training pplx   1.464 | validation pplx   1.407
epoch  4 | training pplx   1.429 | validation pplx   1.397
epoch  5 | training pplx   1.417 | validation pplx   1.388
epoch  6 | training pplx   1.411 | validation pplx   1.395
epoch  7 | training pplx   1.407 | validation pplx   1.383
epoch  8 | training pplx   1.406 | validation pplx   1.380
epoch  9 | training pplx   1.401 | validation pplx   1.383
epoch 10 | training pplx   1.400 | validation pplx   1.381
epoch 11 | training pplx   1.402 | validation pplx   1.381
epoch 12 | training pplx   1.401 | validation pplx   1.384
epoch 13 | training pplx   1.399 | validation pplx   1.385
epoch 14 | training pplx   1.397 | validation pplx   1.380
epoch 15 | training pplx   1.399 | validation pplx   1.381
epoch 16 | training pplx   1.396 | validation pplx   1.389
epoch 17 | training pplx   1.397 | validation pplx   1.3

In [21]:
val_ppl = rnn.evaluate(validation_tensors, collate_fn=collate, n=200, verbose=True)

Average perplexity over 200 runs: 1.52


- an average perplexity value of `1.52` means that the model is on average choosing between 1.52 tokens over 200 runs, meaning that it is mostly certain when choosing the next block 

4. Saving to `.pth` file

In [22]:
rnn.save("lstm.pth")

- all classes and processes created in this notebook is stored within the files `lstm.py` and `trainlstm.py` 

```python



